<a href="https://colab.research.google.com/github/jackeytw886/2026sheep/blob/main/flux_character_lora_trainer_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 靜靜水豚與森林夥伴：Flux 角色一致性 LoRA 訓練站（Google Colab 版）

本筆記本專為 **Google Colab（推薦 A100 40GB/80GB，亦相容 L4 24GB）** 設計，整合 [Ostris ai-toolkit](https://github.com/ostris/ai-toolkit) 與 Google Drive 持久化儲存，讓你用下拉選單一鍵切換訓練以下角色與畫風：

| 角色 / 項目 | 預設觸發詞 | Drive 素材資料夾 |
| :--- | :--- | :--- |
| 靜靜水豚 | `jingcapy` | `AI_Studio/datasets/01_jing_capybara` |
| 台灣黑熊 | `fpart_bear` | `AI_Studio/datasets/02_black_bear` |
| 山羌 | `fpart_muntjac` | `AI_Studio/datasets/03_muntjac` |
| 白鼻心 | `fpart_civet` | `AI_Studio/datasets/04_masked_civet` |
| 穿山甲 | `fpart_pangolin` | `AI_Studio/datasets/05_pangolin` |
| 領角鴞 | `fpart_owl` | `AI_Studio/datasets/06_scops_owl` |
| 家族統一畫風（選用） | `forest_ip_style` | `AI_Studio/datasets/00_forest_ip_style` |

### 開始前請先確認 3 件事
1. **切換 GPU：** 上方選單點選「執行階段」→「變更執行階段類型」→ 選擇 **A100 GPU**（訓練最快，約 25–45 分鐘）或 **L4 GPU**（24 GB VRAM，較省點數但時間較長）。
2. **設定 Hugging Face 金鑰：** 點選左側鑰匙圖示「密鑰（Secrets）」，新增名稱 `HF_TOKEN` 並貼上你的 Hugging Face Read Token（請先至 [black-forest-labs/FLUX.1-dev](https://huggingface.co/black-forest-labs/FLUX.1-dev) 頁面點選同意授權條款）。
3. **準備素材：** 將 15–25 張角色設計稿（PNG/JPG）與同名 `.txt` 英文標註檔，放入對應的 Google Drive 資料夾（或打包成單一 `.zip` 放入該資料夾，本腳本會自動解壓到本地 SSD 加速讀取）。

In [ ]:
#@title 1. 檢查 GPU 規格並掛載 Google Drive（自動建立 6 角色資料夾）
import os
import subprocess
from google.colab import drive

print("=== GPU 狀態檢查 ===")
subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv"], check=False)

drive.mount('/content/drive')

DRIVE_ROOT = "/content/drive/MyDrive/AI_Studio"
DATASET_FOLDERS = [
    "00_forest_ip_style",
    "01_jing_capybara",
    "02_black_bear",
    "03_muntjac",
    "04_masked_civet",
    "05_pangolin",
    "06_scops_owl",
]

for sub in ["loras", "outputs", "workflows", "configs"]:
    os.makedirs(f"{DRIVE_ROOT}/{sub}", exist_ok=True)

for folder in DATASET_FOLDERS:
    os.makedirs(f"{DRIVE_ROOT}/datasets/{folder}", exist_ok=True)

print(f"\n已完成 Google Drive 目錄初始化：{DRIVE_ROOT}")
print("請將各角色的圖片與 .txt 標註檔放入對應的 datasets/ 子資料夾後，再執行下一個儲存格。")

=== GPU 狀態檢查 ===
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

已完成 Google Drive 目錄初始化：/content/drive/MyDrive/AI_Studio
請將各角色的圖片與 .txt 標註檔放入對應的 datasets/ 子資料夾後，再執行下一個儲存格。


In [ ]:
#@title 2. 選擇要訓練的角色與參數設定
TARGET_CHARACTER = "01_jing_capybara | 靜靜水豚 (jingcapy)" #@param ["01_jing_capybara | 靜靜水豚 (jingcapy)", "02_black_bear | 台灣黑熊 (fpart_bear)", "03_muntjac | 山羌 (fpart_muntjac)", "04_masked_civet | 白鼻心 (fpart_civet)", "05_pangolin | 穿山甲 (fpart_pangolin)", "06_scops_owl | 領角鴞 (fpart_owl)", "00_forest_ip_style | 家族統一畫風 (forest_ip_style)"]
VERSION_TAG = "v1" #@param {type:"string"}
TOTAL_STEPS = 1250 #@param {type:"slider", min:500, max:2000, step:250}
LORA_RANK = 16 #@param [16, 32] {type:"raw"}
LEARNING_RATE = 1e-4 #@param [5e-5, 1e-4, 2e-4] {type:"raw"}
GPU_PROFILE = "A100 (40GB/80GB - 高速)" #@param ["A100 (40GB/80GB - 高速)", "L4 (24GB - 開啟低顯存量化)"]
AUTO_DISCONNECT_AFTER_TRAIN = True #@param {type:"boolean"}

CHAR_PRESETS = {
    "01_jing_capybara": {
        "trigger": "jingcapy",
        "en_species": "capybara",
        "prompts": [
            "jingcapy, standing on a sunlit forest trail, friendly smile, full body, storybook illustration",
            "jingcapy, holding a wooden exhibition signpost, clean white background, character sheet style",
            "jingcapy, sitting near a glowing warm lantern at dusk, cozy woodland atmosphere",
            "a cute capybara standing near a river, storybook illustration"
        ]
    },
    "02_black_bear": {
        "trigger": "fpart_bear",
        "en_species": "Formosan black bear",
        "prompts": [
            "fpart_bear, waving happily in a misty green forest, full body, storybook illustration",
            "fpart_bear, holding a small notebook and pencil, clean white background, character sheet style",
            "fpart_bear, sitting beside a campfire at night, warm soft lighting",
            "a black bear walking in the woods, storybook illustration"
        ]
    },
    "03_muntjac": {
        "trigger": "fpart_muntjac",
        "en_species": "muntjac deer",
        "prompts": [
            "fpart_muntjac, standing gracefully among forest ferns, curious expression, full body",
            "fpart_muntjac, pointing at an educational board, clean white background, character sheet style",
            "fpart_muntjac, peeking from behind a mossy tree trunk, warm morning sunlight",
            "a small deer in a forest, storybook illustration"
        ]
    },
    "04_masked_civet": {
        "trigger": "fpart_civet",
        "en_species": "masked palm civet",
        "prompts": [
            "fpart_civet, perched playfully on a tree branch, cheerful expression, full body",
            "fpart_civet, holding a glowing magnifying glass, clean white background, character sheet style",
            "fpart_civet, exploring a moonlit woodland trail, soft lantern light",
            "a civet climbing a tree branch, storybook illustration"
        ]
    },
    "05_pangolin": {
        "trigger": "fpart_pangolin",
        "en_species": "pangolin",
        "prompts": [
            "fpart_pangolin, walking on a mossy forest floor, gentle smile, full body",
            "fpart_pangolin, holding a tiny potted seedling, clean white background, character sheet style",
            "fpart_pangolin, curled playfully near glowing fireflies at twilight",
            "a pangolin walking on grass, storybook illustration"
        ]
    },
    "06_scops_owl": {
        "trigger": "fpart_owl",
        "en_species": "collared scops owl",
        "prompts": [
            "fpart_owl, perched on a wooden signpost with wide curious eyes, full body",
            "fpart_owl, wearing a tiny explorer scarf and holding a map, clean white background",
            "fpart_owl, flying gently above glowing forest lanterns at night",
            "a small owl perched on a branch, storybook illustration"
        ]
    },
    "00_forest_ip_style": {
        "trigger": "forest_ip_style",
        "en_species": "forest exhibition scene",
        "prompts": [
            "forest_ip_style, cozy wooden interpretive pavilion in a misty Taiwanese mountain forest, warm lantern light",
            "forest_ip_style, interactive ecological trail map board surrounded by ferns and fireflies",
            "forest_ip_style, set of botanical icons and woodland props on clean white background",
            "a wooden pavilion in a green forest, watercolor illustration"
        ]
    }
}

FOLDER_KEY = TARGET_CHARACTER.split(" | ")[0].strip()
PRESET = CHAR_PRESETS[FOLDER_KEY]
TRIGGER_WORD = PRESET["trigger"]
RUN_NAME = f"{TRIGGER_WORD}_flux_{VERSION_TAG}"
DRIVE_DATASET_DIR = f"/content/drive/MyDrive/AI_Studio/datasets/{FOLDER_KEY}"
LOCAL_DATASET_DIR = f"/content/dataset/{FOLDER_KEY}"
DRIVE_LORA_OUT = "/content/drive/MyDrive/AI_Studio/loras"
LOW_VRAM_MODE = GPU_PROFILE.startswith("L4")

print(f"訓練專案名稱：{RUN_NAME}")
print(f"觸發詞：{TRIGGER_WORD}")
print(f"步數：{TOTAL_STEPS} | Rank：{LORA_RANK} | LR：{LEARNING_RATE} | Low VRAM 模式：{LOW_VRAM_MODE}")
print(f"素材來源：{DRIVE_DATASET_DIR}")
print(f"模型輸出：{DRIVE_LORA_OUT}/{RUN_NAME}")

訓練專案名稱：jingcapy_flux_v1
觸發詞：jingcapy
步數：1250 | Rank：16 | LR：0.0001 | Low VRAM 模式：False
素材來源：/content/drive/MyDrive/AI_Studio/datasets/01_jing_capybara
模型輸出：/content/drive/MyDrive/AI_Studio/loras/jingcapy_flux_v1


In [ ]:
#@title 3. 複製/解壓素材到本地 SSD 並自動健檢（圖片數與標註檔檢查）
import glob
import os
import shutil
import zipfile

os.makedirs(LOCAL_DATASET_DIR, exist_ok=True)
# 清空舊的本地快取避免混檔
for f in os.listdir(LOCAL_DATASET_DIR):
    p = os.path.join(LOCAL_DATASET_DIR, f)
    if os.path.isdir(p):
        shutil.rmtree(p)
    else:
        os.remove(p)

zip_files = glob.glob(os.path.join(DRIVE_DATASET_DIR, "*.zip"))
if zip_files:
    print(f"偵測到壓縮檔，正在解壓至本地 SSD：{zip_files[0]}")
    with zipfile.ZipFile(zip_files[0], "r") as zf:
        zf.extractall(LOCAL_DATASET_DIR)
else:
    print(f"正在從 Google Drive 複製素材到本地 SSD：{LOCAL_DATASET_DIR}")
    for item in os.listdir(DRIVE_DATASET_DIR):
        src = os.path.join(DRIVE_DATASET_DIR, item)
        dst = os.path.join(LOCAL_DATASET_DIR, item)
        if os.path.isfile(src):
            shutil.copy2(src, dst)

# 若解壓後多包了一層子資料夾，自動攤平
subitems = os.listdir(LOCAL_DATASET_DIR)
if len(subitems) == 1 and os.path.isdir(os.path.join(LOCAL_DATASET_DIR, subitems[0])):
    nested = os.path.join(LOCAL_DATASET_DIR, subitems[0])
    for f in os.listdir(nested):
        shutil.move(os.path.join(nested, f), os.path.join(LOCAL_DATASET_DIR, f))
    os.rmdir(nested)

img_exts = (".png", ".jpg", ".jpeg", ".webp")
images = sorted([f for f in os.listdir(LOCAL_DATASET_DIR) if f.lower().endswith(img_exts)])
missing_txt = []
missing_trigger = []

for img in images:
    stem = os.path.splitext(img)[0]
    txt_path = os.path.join(LOCAL_DATASET_DIR, stem + ".txt")
    if not os.path.exists(txt_path):
        missing_txt.append(img)
        # 自動補上最小預設標註，避免訓練中斷
        with open(txt_path, "w", encoding="utf-8") as fp:
            fp.write(f"{TRIGGER_WORD}, storybook character illustration\n")
    else:
        content = open(txt_path, encoding="utf-8").read().strip()
        if TRIGGER_WORD not in content:
            missing_trigger.append(img)

print(f"\n=== 素材健檢結果 ===")
print(f"總圖片數：{len(images)} 張（建議 15–25 張）")
if len(images) == 0:
    raise RuntimeError(f"找不到圖片！請先將角色設計稿上傳到 {DRIVE_DATASET_DIR}")
if missing_txt:
    print(f"[提示] 有 {len(missing_txt)} 張圖缺少 .txt 標註檔，已自動補寫基本觸發詞（建議補齊動作與背景描述效果更好）。")
if missing_trigger:
    print(f"[警告] 有 {len(missing_trigger)} 個標註檔未包含觸發詞 '{TRIGGER_WORD}'：{missing_trigger[:5]}")
else:
    print(f"[通過] 所有標註檔皆已包含觸發詞 '{TRIGGER_WORD}'。")

正在從 Google Drive 複製素材到本地 SSD：/content/dataset/01_jing_capybara

=== 素材健檢結果 ===
總圖片數：0 張（建議 15–25 張）


RuntimeError: 找不到圖片！請先將角色設計稿上傳到 /content/drive/MyDrive/AI_Studio/datasets/01_jing_capybara

In [ ]:
#@title 4. 安裝 Ostris ai-toolkit 並登入 Hugging Face（啟用高速下載）
import os
from google.colab import userdata

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

try:
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    hf_token = None

if not hf_token:
    raise ValueError("請先在 Colab 左側「密鑰（Secrets）」新增 HF_TOKEN，並開啟此筆記本的存取權限。")

os.environ["HF_TOKEN"] = hf_token

if not os.path.exists("/content/ai-toolkit"):
    !git clone https://github.com/ostris/ai-toolkit.git /content/ai-toolkit
    %cd /content/ai-toolkit
    !git submodule update --init --recursive
    !pip install -q --no-cache-dir -r requirements.txt hf_transfer pyyaml
else:
    %cd /content/ai-toolkit
    print("ai-toolkit 已存在，略過重複安裝。")

!huggingface-cli login --token "$HF_TOKEN" --add-to-git-credential
print("環境安裝與 Hugging Face 驗證完成！")

In [ ]:
#@title 5. 自動產生訓練設定檔（YAML）並備份至 Google Drive
import yaml

config = {
    "job": "extension",
    "config": {
        "name": RUN_NAME,
        "process": [
            {
                "type": "sd_trainer",
                "training_folder": DRIVE_LORA_OUT,
                "device": "cuda:0",
                "trigger_word": TRIGGER_WORD,
                "network": {
                    "type": "lora",
                    "linear": int(LORA_RANK),
                    "linear_alpha": int(LORA_RANK),
                },
                "save": {
                    "dtype": "float16",
                    "save_every": 250,
                    "max_step_saves_to_keep": 4,
                    "push_to_hub": False,
                },
                "datasets": [
                    {
                        "folder_path": LOCAL_DATASET_DIR,
                        "caption_ext": "txt",
                        "caption_dropout_rate": 0.05,
                        "shuffle_tokens": False,
                        "cache_latents_to_disk": True,
                        "resolution": [512, 768, 1024],
                    }
                ],
                "train": {
                    "batch_size": 1,
                    "steps": int(TOTAL_STEPS),
                    "gradient_accumulation_steps": 1,
                    "train_unet": True,
                    "train_text_encoder": False,
                    "gradient_checkpointing": True,
                    "noise_scheduler": "flowmatch",
                    "optimizer": "adamw8bit",
                    "lr": float(LEARNING_RATE),
                    "ema_config": {
                        "use_ema": True,
                        "ema_decay": 0.99,
                    },
                    "dtype": "bf16",
                },
                "model": {
                    "name_or_path": "black-forest-labs/FLUX.1-dev",
                    "is_flux": True,
                    "quantize": True,
                    "low_vram": bool(LOW_VRAM_MODE),
                },
                "sample": {
                    "sampler": "flowmatch",
                    "sample_every": 250,
                    "width": 1024,
                    "height": 1024,
                    "prompts": PRESET["prompts"],
                    "neg": "",
                    "seed": 42,
                    "walk_seed": True,
                    "guidance_scale": 4,
                    "sample_steps": 20,
                },
            }
        ],
    },
}

CONFIG_PATH = f"/content/ai-toolkit/config/{RUN_NAME}.yaml"
DRIVE_CONFIG_BACKUP = f"/content/drive/MyDrive/AI_Studio/configs/{RUN_NAME}.yaml"

os.makedirs(os.path.dirname(CONFIG_PATH), exist_ok=True)
with open(CONFIG_PATH, "w", encoding="utf-8") as f:
    yaml.safe_dump(config, f, sort_keys=False, allow_unicode=True)

shutil.copy2(CONFIG_PATH, DRIVE_CONFIG_BACKUP)
print(f"已產生設定檔：{CONFIG_PATH}")
print(f"已同步備份至：{DRIVE_CONFIG_BACKUP}")

In [ ]:
#@title 6. 啟動 LoRA 訓練（每 250 步自動存檔回 Drive，訓完可自動斷線省點數）
import time

start_ts = time.time()
%cd /content/ai-toolkit
!python run.py "{CONFIG_PATH}"
elapsed_min = (time.time() - start_ts) / 60.0

print(f"\n=== 訓練完成！總耗時：{elapsed_min:.1f} 分鐘 ===")
print(f"模型權重與預覽圖已存至：{DRIVE_LORA_OUT}/{RUN_NAME}")

if AUTO_DISCONNECT_AFTER_TRAIN:
    print("已開啟自動斷線省點數模式：15 秒後自動釋放 GPU 執行階段...")
    time.sleep(15)
    from google.colab import runtime
    runtime.unassign()

In [ ]:
#@title 7. （選用）檢視各階段預覽圖，挑選最佳步數權重
# 若上面開啟了自動斷線，可在免費 CPU 執行階段執行本格直接瀏覽 Google Drive 裡的預覽圖
import glob
import os
from IPython.display import display, HTML
from PIL import Image

sample_dir = f"/content/drive/MyDrive/AI_Studio/loras/{RUN_NAME}/samples"
sample_imgs = sorted(glob.glob(os.path.join(sample_dir, "*.jpg")) + glob.glob(os.path.join(sample_dir, "*.png")))

if not sample_imgs:
    print(f"尚未在 {sample_dir} 找到預覽圖。")
else:
    print(f"共找到 {len(sample_imgs)} 張預覽圖，以下顯示最新 8 張（建議對比第 750 / 1000 / 1250 步，挑選最像原稿且背景自然的一版）：")
    for img_path in sample_imgs[-8:]:
        print(os.path.basename(img_path))
        im = Image.open(img_path)
        im.thumbnail((512, 512))
        display(im)